In [61]:
import torch
import triton
import triton.language as tl

@triton.jit
def fused_add_gelu_kernel(
    x_ptr,
    bias_ptr,
    y_ptr,
    n_elements,
    BLOCK_SIZE:tl.constexpr,
):
  pid=tl.program_id(0)
  one_block=pid*BLOCK_SIZE

  offsets=one_block+tl.arange(0, BLOCK_SIZE)
  mask=offsets<n_elements

  x=tl.load(x_ptr+offsets, mask=mask)
  bias=tl.load(bias_ptr+offsets, mask=mask)

  intermediate_sum=x+bias

  gelu=0.5*intermediate_sum*(1+(2*tl.sigmoid(2*0.79788456*(intermediate_sum+0.044715*intermediate_sum*intermediate_sum*intermediate_sum))-1))

  tl.store(y_ptr+offsets,gelu,mask=mask)



In [62]:
import torch
def fused_add_gelu(x, bias):
  y=torch.empty_like(x)
  n_elements=x.numel()

  grid=lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)

  fused_add_gelu_kernel[grid](
      x,
      bias,
      y,
      n_elements,
      BLOCK_SIZE=1024,
  )
  return y


In [64]:
x = torch.randn(10_000_000, device="cuda")
bias = torch.randn(10_000_000, device="cuda")

y_torch = torch.nn.functional.gelu(x + bias, approximate="tanh")
y_triton = fused_add_gelu(x, bias)

torch.cuda.synchronize()

# Add atol and rtol parameters
print(torch.allclose(y_torch, y_triton, atol=1e-5, rtol=1e-5))

True


In [65]:
import time

torch.cuda.synchronize()
start = time.perf_counter()

for _ in range(100):
    y = torch.nn.functional.gelu(x + bias, approximate="tanh")

torch.cuda.synchronize()
end = time.perf_counter()

print("PyTorch:", (end - start) / 100)

PyTorch: 0.0008321341999999276


In [66]:
torch.cuda.synchronize()
start = time.perf_counter()

for _ in range(100):
    fused_add_gelu_kernel[grid](
        x,
        bias,
        y_triton,
        x.numel(),
        BLOCK_SIZE=1024,
    )

torch.cuda.synchronize()
end = time.perf_counter()

print("Triton:", (end - start) / 100)

Triton: 4.036407999592484e-05
